# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def select_uids_by_class(df, sample_number, class_col='Class', uid_col='uid', random_state=42):
    np.random.seed(random_state)
    uids_by_class = {}
    for c in df[class_col].unique():
        uids = df[df[class_col]==c][uid_col].unique()
        n_select = min(sample_number, len(uids))
        selected_uids = np.random.choice(uids, n_select, replace=False)
        uids_by_class[c] = set(selected_uids)
    return uids_by_class

In [6]:
def filter_by_selected_uids(df, uids_by_class, class_col='Class', uid_col='uid'):
    mask = np.zeros(len(df), dtype=bool)
    for c, uids in uids_by_class.items():
        mask |= ((df[class_col]==c) & (df[uid_col].isin(uids)))
    return df[mask]

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def train_test_split_by_uid(df, test_size=0.2, class_col='Class', uid_col='uid', random_state=42):
    groups = df[uid_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, df[class_col], groups))
    return df.iloc[train_idx], df.iloc[test_idx]

## Initial Setup

In [8]:
logtxt = ""

In [9]:
# Set experiment configs
test_name = "new_LightGBM_Fine_Tune_20"
random_state = 42
test_size = 0.2
device_type = "cpu" # or gpu
n_jobs = -1
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)

logtxt += "\nSet experiment configs\n"
logtxt += f"test_name: {test_name}\n"
logtxt += f"random_state: {random_state}\n"
logtxt += f"test_size: {test_size}\n"
logtxt += f"device_type: {device_type}\n"
logtxt += f"n_jobs: {n_jobs}\n"
logtxt += f"path_save: {path_save}\n"
logtxt += "\n"


- Source to Consider

In [10]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	"Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
]
logtxt += f"\nSources to consider: {sources_to_consider}\n"

In [11]:
path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_40.csv')

logtxt += f"\nBalanced Data Set\n"

# Data

In [12]:
columns_to_use = list(data_dtype_dict_20.keys())

In [13]:
data = pd.read_csv(
    path_data,
    engine='c', 
    # usecols=columns_to_use,
    # dtype=data_dtype_dict,
)

data['uid'] = data['uid'].astype(str)
data['Class'] = data['Class'].astype(str)

uids = data['uid'].values
classes = data['Class'].values

print(f"Balanced Data: {len(data)}")

logtxt += f"Balanced Data: {len(data)}\n"

indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"


Balanced Data: 282753
10 sources to consider: 282753


- Training and Test Data

In [14]:
# Step 0: Define GV group and label
gv_classes = ["Nova", "M dwarf", "CV", "LBV"]
unified_label = "GV"

# Step 1: Initialize list to collect subsampled data
gv_subsample_list = []

# Step 2: For each GV class, sample a subset of rows per uid
for cls in gv_classes:
    cls_data = data[data['Class'] == cls]
    # Group by uid and sample {frac}% of each uid's samples (at least 1)
    sampled_data = cls_data.groupby('uid').apply(
        lambda df: df.sample(frac=0.25, random_state=42) if len(df) > 1 else df
    ).reset_index(drop=True)

    # Replace label to unified GV label
    sampled_data = sampled_data.copy()
    sampled_data['Class'] = unified_label
    gv_subsample_list.append(sampled_data)

# Step 3: Concatenate subsampled GV data
gv_data = pd.concat(gv_subsample_list, ignore_index=True)

# Step 4: Keep other classes unchanged
non_gv_data = data[~data['Class'].isin(gv_classes)]

# Step 5: Merge into final data
data = pd.concat([non_gv_data, gv_data], ignore_index=True)

# Step 6: Redefine sources_to_consider
sources_to_consider = [
    "AGN", 
    "Ia", 
    "II", 
    "Ibc", 
    "TDE", 
    "SLSN", 
    unified_label,
]
logtxt += f"\nSources to consider (with GV sampled): {sources_to_consider}\n"

# Step 7: Filter data again
indx_type_to_consider = np.where(
    np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)
print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]
uids = data['uid'].values
logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n\n"

# Step 8: Train/test split & encoding
X = data.drop(columns=['Sample_ID', 'Class', 'uid'])
y = data['Class']
X.fillna(-99, inplace=True)


7 sources to consider: 201000


In [15]:
# - Split features/target
# X = data.drop(columns=['Sample_ID', 'Class', 'uid'])
# y = data['Class']
# X.fillna(-99, inplace=True)

# - Split into train/test using GroupShuffleSplit by uid
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(X, y, groups=uids))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# - Label encode class for ML
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
print("Balanced: Class mapping:", class_names)

# Tets\\sts
classifier_type = 'normal_class_classifier'
model_param_config = model_config[classifier_type][device_type]

import lightgbm as lgb
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)


Balanced: Class mapping: ['AGN' 'GV' 'II' 'Ia' 'Ibc' 'SLSN' 'TDE']


In [16]:
print(len(X), len(y), len(uids))

201000 201000 201000


In [17]:
del data

# Fine Tuning

In [18]:
import time
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score
import optuna
from lightgbm import LGBMClassifier

def objective(trial):
    params = {
    'num_leaves': trial.suggest_int('num_leaves', 16, 64),
    'n_estimators': trial.suggest_int('n_estimators', 100, 500),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 20), # prevent overfitting
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    'random_state': random_state,
    'force_row_wise': True,
    'verbose': -1,
    'num_threads': n_jobs,
    # 'device': 'gpu',
    }
    print("Optimization Start with parameter ranges:")
    print(params)
    #
    callbacks = [
        lgb.early_stopping(stopping_rounds=30),
        ]
    #
    model = LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        #
        eval_set=[(X_test, y_test)],
        eval_metric='multi_logloss',
        callbacks=callbacks,
        )
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred, average='macro')

In [19]:
number_of_trials = 50

best_parameter_filename = f"{path_save}/best_params_n{number_of_trials}.yaml"

if not os.path.exists(best_parameter_filename):

    t0 = time.time()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=number_of_trials)
    delt = time.time() - t0
    print(f"It takes {delt/60:.1f} min")
    print(f"Best Value = {study.best_value}, Best Parameters = {study.best_params}")
    print("Best trial:")
    trial = study.best_trial

    print("Value: ", trial.value)
    print("Params: ")
    for key, value in trial.params.items():
        print(f"{key}: {value}")

    from optuna.visualization import plot_optimization_history, plot_param_importances
    import hiplot as hip

    plot_optimization_history(study)
    data = [dict(trial.params, value=trial.value) for trial in study.trials]
    hip.Experiment.from_iterable(data).display()
    best_params = trial.params

else:
    with open(best_parameter_filename, 'r', encoding='utf-8') as file:
        best_params = yaml.safe_load(file)


best_params['random_state'] = 42
best_params['num_threads'] = n_jobs
#

import yaml
with open(best_parameter_filename, "w") as f:
    yaml.dump(best_params, f, default_flow_style=False)


[I 2025-06-05 07:50:07,199] A new study created in memory with name: no-name-00fc15f6-885b-409e-8ec7-5deb31d4357e


Optimization Start with parameter ranges:
{'num_leaves': 18, 'n_estimators': 270, 'min_child_weight': 11, 'learning_rate': 0.03346822673414524, 'colsample_bytree': 0.6822857647143585, 'random_state': 42, 'force_row_wise': True, 'verbose': -1, 'num_threads': -1}
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[220]	valid_0's multi_logloss: 0.641198


[I 2025-06-05 07:58:04,096] Trial 0 finished with value: 0.7930106759612775 and parameters: {'num_leaves': 18, 'n_estimators': 270, 'min_child_weight': 11, 'learning_rate': 0.03346822673414524, 'colsample_bytree': 0.6822857647143585}. Best is trial 0 with value: 0.7930106759612775.


Optimization Start with parameter ranges:
{'num_leaves': 30, 'n_estimators': 345, 'min_child_weight': 11, 'learning_rate': 0.030200046672715412, 'colsample_bytree': 0.71950309536378, 'random_state': 42, 'force_row_wise': True, 'verbose': -1, 'num_threads': -1}
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[165]	valid_0's multi_logloss: 0.635646


[I 2025-06-05 08:04:48,655] Trial 1 finished with value: 0.7936035893730752 and parameters: {'num_leaves': 30, 'n_estimators': 345, 'min_child_weight': 11, 'learning_rate': 0.030200046672715412, 'colsample_bytree': 0.71950309536378}. Best is trial 1 with value: 0.7936035893730752.


...

Optimization Start with parameter ranges:
{'num_leaves': 61, 'n_estimators': 242, 'min_child_weight': 13, 'learning_rate': 0.10475239355157495, 'colsample_bytree': 0.8015257058854665, 'random_state': 42, 'force_row_wise': True, 'verbose': -1, 'num_threads': -1}
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[32]	valid_0's multi_logloss: 0.64578


[I 2025-06-05 08:58:29,678] Trial 18 finished with value: 0.7882628095102792 and parameters: {'num_leaves': 61, 'n_estimators': 242, 'min_child_weight': 13, 'learning_rate': 0.10475239355157495, 'colsample_bytree': 0.8015257058854665}. Best is trial 17 with value: 0.79534610036433.


Optimization Start with parameter ranges:
{'num_leaves': 64, 'n_estimators': 341, 'min_child_weight': 6, 'learning_rate': 0.04557749994117585, 'colsample_bytree': 0.8001540541917932, 'random_state': 42, 'force_row_wise': True, 'verbose': -1, 'num_threads': -1}
Training until validation scores don't improve for 30 rounds


[W 2025-06-05 09:01:10,319] Trial 19 failed with parameters: {'num_leaves': 64, 'n_estimators': 341, 'min_child_weight': 6, 'learning_rate': 0.04557749994117585, 'colsample_bytree': 0.8001540541917932} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/gpaek/.conda/envs/sedml/lib/python3.8/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3130840/31983526.py", line 28, in objective
    model.fit(
  File "/home/gpaek/.conda/envs/sedml/lib/python3.8/site-packages/lightgbm/sklearn.py", line 1284, in fit
    super().fit(
  File "/home/gpaek/.conda/envs/sedml/lib/python3.8/site-packages/lightgbm/sklearn.py", line 955, in fit
    self._Booster = train(
  File "/home/gpaek/.conda/envs/sedml/lib/python3.8/site-packages/lightgbm/engine.py", line 307, in train
    booster.update(fobj=fobj)
  File "/home/gpaek/.conda/envs/sedml/lib/python3.8/site-packages/lightgbm/basic.py",

KeyboardInterrupt: 

# Take Best Parameter 

In [ ]:
best_params['force_row_wise'] = True
best_params

{'colsample_bytree': 0.6824013531577555,
 'learning_rate': 0.05132162353818825,
 'min_child_weight': 7,
 'n_estimators': 382,
 'num_leaves': 43,
 'num_threads': -1,
 'random_state': 42,
 'force_row_wise': True}

In [ ]:
# best_model = LGBMClassifier(**best_params)
# best_model.fit(
#     X_train, y_train,
#     )
# y_pred = best_model.predict(X_test)
# best_model.save_model(f"{path_save}/best_lightgbm_model.txt")

# Results - 5-fold CV

In [ ]:

from lightgbm import LGBMClassifier  # 예시로 LGBM 사용
import joblib  # for XGBoost and LightGBM
from sklearn.metrics import classification_report
import pandas as pd
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report

# 저장 경로 생성
os.makedirs(path_save, exist_ok=True)

df_cm_filename = os.path.join(path_save, f'confusion_matrix.csv')

# feature_to_use = rubin_added_feature_dict[filte]
# X_sub = X[feature_to_use]
# X_train_sub = X_train[feature_to_use]
# X_test_sub = X_test[feature_to_use]

if not os.path.exists(df_cm_filename):
    
    #
    print(f"Start Training")
    callbacks = [
        lgb.early_stopping(stopping_rounds=30),
        ]
    #
    print(f"Parameters: {best_params}")
    best_model = LGBMClassifier(**best_params)
    best_model.fit(
        X_train, y_train,
        #
        eval_set=[(X_test, y_test)],
        eval_metric='multi_logloss',
        callbacks=callbacks,
        # force_col_wise=True,
        )
    y_pred = best_model.predict(X_test)
    # best_model.save_model(f"{path_save}/best_lightgbm_model.txt")
    #
    
    # # ---------------------
    # # CatBoost 저장
    # # ---------------------
    # cat_model.save_model(os.path.join(path_save, "best_catboost_model.txt"))
    
    # # ---------------------
    # # XGBoost 저장
    # # ---------------------
    # xgb_model.save_model(os.path.join(path_save, "best_xgboost_model.json"))  # .json preferred over .txt
    
    # # ---------------------
    # # LightGBM 저장
    # # ---------------------
    joblib.dump(best_model, os.path.join(path_save, f"lightgbm_7DT.pkl"))
    #

    print(f"Get Classification Report")
    report_dict = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    
    # 2. accuracy는 scalar → 딕셔너리로 따로 저장
    acc_value = report_dict.pop("accuracy")
    
    # 3. DataFrame으로 변환
    report_df = pd.DataFrame(report_dict).T  # class별 성능 포함
    
    # 4. accuracy row 추가 (f1-score 칸만 채움)
    report_df.loc["accuracy"] = [None, None, acc_value, report_df["support"].sum()]
    report_df.to_csv(os.path.join(path_save, f"classification_report.csv"))

    print(f"Get Confusion Matrix")
    plot_confusion_matrix(y_test, y_pred, class_names)
    plt.savefig(os.path.join(path_save, f'confusion_matrix.png'))
    # plt.close()
    
    cm = confusion_matrix(y_test, y_pred)
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    df_cm.to_csv(df_cm_filename)
else:
    print(df_cm_filename, "exists")



/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/confusion_matrix.csv exists


In [ ]:
# 설정
class_labels = label_encoder.classes_
gkf = GroupKFold(n_splits=5)

cv_macro_f1_scores = []
cv_per_class_f1_scores = []


print(f"Calculate 5-Fold Cross-Validation Values")

# Perform Grouped K-Fold Cross-Validation
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups=uids)):

    macro_f1_df_filename = os.path.join(path_save, f"cv_macro_f1_scores_{fold}.csv")
    per_class_f1_df_filename = os.path.join(path_save, f"cv_per_class_f1_scores_{fold}.csv")

    if (not os.path.exists(macro_f1_df_filename)) & ((not os.path.exists(per_class_f1_df_filename))):
        
        X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
        y_train_cv, y_val_cv = y_encoded[train_idx], y_encoded[val_idx]
    
        callbacks = [
            lgb.early_stopping(stopping_rounds=30),
            ]
        #
        model_cv = LGBMClassifier(**best_params)
        model_cv.fit(
            X_train_cv, y_train_cv,
            #
            eval_set=[(X_val_cv, y_val_cv)],
            eval_metric='multi_logloss',
            callbacks=callbacks,
            # force_col_wise=True,
            )
   
        y_pred = model_cv.predict(X_val_cv)
    
        # === 1. Macro F1 저장 ===
        macro_f1 = f1_score(y_val_cv, y_pred, average='macro')
        cv_macro_f1_scores.append({'fold': fold, 'f1_macro': macro_f1})
    
        # === 2. Per-class F1 저장 ===
        report = classification_report(y_val_cv, y_pred, output_dict=True, zero_division=0)
        per_class_f1 = {label: report[str(i)]['f1-score'] for i, label in enumerate(class_labels)}
        per_class_f1['fold'] = fold
        cv_per_class_f1_scores.append(per_class_f1)
    
        # === 3. 모델 저장 ===
        model_path = os.path.join(path_save, f"model_fold{fold}.pkl")
        joblib.dump(model_cv, model_path)
    
        # === 4. 리포트 출력 ===
        print(f"[Fold {fold}] f1_macro = {macro_f1:.3f}")
        
        # === 5. 최종 결과 저장 ===
        macro_f1_df = pd.DataFrame(cv_macro_f1_scores)
        per_class_f1_df = pd.DataFrame(cv_per_class_f1_scores)


        macro_f1_df.to_csv(macro_f1_df_filename, index=False)
        per_class_f1_df.to_csv(per_class_f1_df_filename, index=False)
    else:
        print(macro_f1_df_filename, "exists")
        print(per_class_f1_df_filename, "exists")

Calculate 5-Fold Cross-Validation Values
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_macro_f1_scores_0.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_per_class_f1_scores_0.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_macro_f1_scores_1.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_per_class_f1_scores_1.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_macro_f1_scores_2.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_per_class_f1_scores_2.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_macro_f1_scores_3.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_per_class_f1_scores_3.csv exists
/home/gpaek/SED-Classifier/notebook/../model/new_LightGBM_Fine_Tune/cv_macro_f1_scores_4.csv exists
/home/gpaek/SED-Classifier/notebook/../mode